In [62]:
from skopt import gp_minimize
import traceback
import pickle
from tqdm import tqdm  # Import tqdm for progress bar
from skopt.space import Integer, Real, Categorical
from skopt.utils import use_named_args
import os 
# disable XLA - don't want it running inside Bayesian optimisation loop
os.environ["TF_XLA_FLAGS"] = "--tf_xla_enable_xla_devices=false"
os.environ["TF_ENABLE_XLA"] = "0"
os.environ["CUDA_VISIBLE_DEVICES"] = "1"

import tensorflow as tf
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras import layers
from tensorflow.keras.layers import (
    Conv2D,
    MaxPooling2D,
    Flatten,
    Dense,
    BatchNormalization,
    Dropout,
    GlobalAveragePooling2D,
    Input,
    LayerNormalization,
    MultiHeadAttention,
    Add,
    Embedding,
    Reshape,
)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, CSVLogger
from tensorflow.keras.backend import clear_session
import pandas as pd
import numpy as np
import gc
from datetime import datetime, date
import cProfile
import pstats


tf.config.optimizer.set_jit(False)
tf.config.optimizer.set_experimental_options({
    "layout_optimizer": False
})

tf.config.optimizer.set_experimental_options({
    "layout_optimizer": False
})
gpus = tf.config.list_physical_devices("GPU")
tf.config.experimental.set_memory_growth(gpus[0], True)

print("Num GPUs Available: ", len(tf.config.experimental.list_physical_devices('GPU')))


np.random.seed(779)
tf.random.set_seed(779)

profiler = cProfile.Profile()
profiler.enable()


2026-02-16 16:30:22.559380: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-02-16 16:30:22.614010: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-02-16 16:30:22.614040: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-02-16 16:30:22.614988: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-02-16 16:30:22.623009: I tensorflow/core/platform/cpu_feature_guar

Num GPUs Available:  1


In [63]:

testing = True # Set to True for testing the code quickly
start_time = datetime.now()
today = datetime.now().strftime('%Y-%m-%d')
img_size = 40
Code_run_ID = today + 'run_29_ViT_bays_opt_40px'
batch_size = 32
input_shape = (img_size, img_size, 210)  # Load the data


In [64]:
# path of optimisation data files
training_data_path = '/home/ht21074/Auto_box_apples/Auto_box_apples/Source_folder/Bayesian_optimisation_files/'
# Using the same validation data from training data - 50px for 40px images, and 30px for 20px images, to ensure the same data is used across all models for fair comparison
validation_file_path = '/home/ht21074/Auto_box_apples/Auto_box_apples/Source_folder/Training_data_May2025/50px/all_years/'
save_file_path = '/media/2tbdisk3/data/Haidee/Results/'
    

In [65]:
# data_type = ['brix', 'firmness', 'starch']
data_type = ['starch']

def load_data(data_type):
    if data_type == 'brix':
        spectral_path = '/media/2tbdisk3/data/Haidee/May_2025_spectral_img/Spectral/'

        X_train = np.load(f'{training_data_path}X_train_brix_optimisation_May2025.npy')
        y_train = np.load(f'{training_data_path}Y_train_brix_optimisation_May2025.npy')                     
        X_val= np.load(f'{validation_file_path}X_validate_all_years_Brix_shuffled.npy')                 
        y_val= np.load(f'{validation_file_path}Y_validate_all_years_Brix_shuffled.npy')                
        train_cultivars = np.load(f'{training_data_path}X_encoder_brix_optimisation_May2025.npy')   
        validate_encoder = np.load(f'{validation_file_path}X_validate_all_years_Brix_encoder_shuffled.npy')
        

        X_train = [spectral_path + file for file in X_train]
        X_val = [spectral_path + file for file in X_val]
    elif data_type == 'firmness':
        spectral_path = '/media/2tbdisk3/data/Haidee/May_2025_spectral_img/Spectral/'

        X_train = np.load(f'{training_data_path}X_train_firmness_optimisation_May2025.npy')
        y_train = np.load(f'{training_data_path}Y_train_firmness_optimisation_May2025.npy')                     
        X_val= np.load(f'{validation_file_path}X_validate_all_years_Firmness_shuffled.npy')                  
        y_val= np.load(f'{validation_file_path}Y_validate_all_years_Firmness_shuffled.npy')                
        train_cultivars = np.load(f'{training_data_path}X_encoder_firmness_optimisation_May2025.npy')   
        validate_encoder = np.load(f'{validation_file_path}X_validate_all_years_Firmness_encoder_shuffled.npy')
        X_train = [spectral_path + file for file in X_train]
        X_val = [spectral_path + file for file in X_val]
        
    elif data_type == 'starch':
        spectral_path = '/media/2tbdisk3/data/Haidee/May_2025_spectral_img/Spectral/'

        X_train = np.load(f'{training_data_path}X_train_starch_optimisation_May2025.npy')
        y_train = np.load(f'{training_data_path}Y_train_starch_optimisation_May2025.npy')                     
        X_val= np.load(f'{validation_file_path}X_validate_all_years_Starch_shuffled.npy')                  
        y_val= np.load(f'{validation_file_path}Y_validate_all_years_Starch_shuffled.npy')                
        train_cultivars = np.load(f'{training_data_path}X_encoder_starch_optimisation_May2025.npy')   
        validate_encoder = np.load(f'{validation_file_path}X_validate_all_years_Starch_encoder_shuffled.npy')
        X_train = [spectral_path + file for file in X_train]
        X_val = [spectral_path + file for file in X_val]
    return X_train, y_train, X_val, y_val, train_cultivars, validate_encoder


In [66]:
def data_generator_w_cultivar(file_list, targets, cultivars, batch_size, img_size):
    num_samples = len(file_list)
    missing_files = []  # List of missing files
    while True:  # Infinite loop for generator
        for offset in range(0, num_samples, batch_size):
            # Load the batch of data from file paths
            batch_files = file_list[offset: offset + batch_size]
            batch_data = []
            batch_targets = []
            batch_cultivars = []
            
            # File loading and handling - ensures model runs if file not found
            for i, file in enumerate(batch_files):
                try:
                    data = np.load(file)
                    # print(data.shape)
                    if img_size == 40 or img_size ==20:
                        data_reduced = data[5:-5, 5:-5, :] # Remove 5 pixels from each edge
                        batch_data.append(data_reduced)
                    else:
                        batch_data.append(data)
                    batch_targets.append(targets[offset + i])
                    batch_cultivars.append(cultivars[offset + i])
                except FileNotFoundError:
                    missing_files.append(file)
                    print(f"File not found: {file}. Skipping...")
                    continue
            
            # Convert lists to numpy arrays
            batch_data = np.array(batch_data)  # Shape: (batch_size, 20, 20, 204)
            # print(batch_data.shape)
            batch_targets = np.array(batch_targets)  # Shape: (batch_size,)
            batch_cultivars = np.array(batch_cultivars)  # Shape: (batch_size, 6)
            
            if len(batch_data) == 0:
                continue  # Skip if no data loaded
            
            # Expand cultivar information to match the input data's spatial dimensions
            expanded_cultivars = np.repeat(batch_cultivars[:, np.newaxis, np.newaxis, :], img_size, axis=1)
            expanded_cultivars = np.repeat(expanded_cultivars, img_size, axis=2)
            # print(expanded_cultivars.shape)

            # Concatenate cultivar information with the original data along the last axis
            combined_data = np.concatenate([batch_data, expanded_cultivars], axis=-1)  # Shape: (batch_size, 14, 14, 210)
            
            # Yield the combined data and targets
            yield combined_data, batch_targets

In [67]:


reduce_lr = ReduceLROnPlateau(
    monitor='val_mae',
    factor=0.5,
    patience=10,
    min_lr=1e-6,
    verbose=1,
    mode='min'
)

def mlp(x, hidden_units, dropout_rate):
    for units in hidden_units:
        x = Dense(units, activation=tf.nn.gelu)(x)
        x = Dropout(dropout_rate)(x)
    return x

# def parse_mlp_units(mlp_string):
#     return [int(x) for x in mlp_string.split('-')]



In [68]:
search_space = [
    Integer(2, 10, name = "patch_size"),  # Adjust based on your image size
    Integer(64, 256, name = "projection_dim"),
    Integer(1, 5, name = "transformer_layers"),
    Integer(1, 5, name = "num_heads"),
    Categorical([
        (128, 64),
        (256, 128, 64),
        (256, 128)
    ], name="mlp_head_units"),
    Real(0.1, 0.5, name="dropout_rate")
]



In [69]:
# print(input_shape)
default_params = [5, 128, 4, 5, (256, 128, 64), 0.1]

In [9]:
def create_vit_model(
    patch_size=5,  # Size of patches to extract from the input image
    projection_dim=128,  # Embedding dimension for each patch
    transformer_layers=4,  # Number of Transformer blocks
    num_heads=5,  # Number of attention heads
    mlp_head_units=list([256, 128, 64]),  # Hidden units in the MLP head
    dropout_rate=0.1,  # Dropout rate
):
    global input_shape  

    # print('patch size:', 
    #       patch_size, 'projection dim:', 
    #       projection_dim, 'transformer layers:', 
    #       transformer_layers, 'num heads:', 
    #       num_heads, 'mlp head units:', 
    #       mlp_head_units, 'dropout rate:', 
    #       dropout_rate)
    mlp_head_units = list(mlp_head_units)  # Ensure mlp_head_units is a list

    # print(type(input_shape[0]), type(patch_size))

    # Calculate number of patches
    num_patches = (input_shape[0] // patch_size) * (input_shape[1] // patch_size)
    
    # Create input layer
    inputs = Input(shape=input_shape)
    
    # Create patches using Conv2D and Reshape
    patches = layers.Conv2D(
        filters=projection_dim,
        # kernel_size = patch_size,
        # strides = patch_size,
        kernel_size=(patch_size, patch_size), 
        strides=(patch_size, patch_size),
        padding="valid",
    )(inputs)
    patches = layers.Reshape((-1, projection_dim))(patches)
    
    # Add positional embeddings
    positions = tf.range(start=0, limit=num_patches, delta=1)
    position_embedding = layers.Embedding(
        input_dim=num_patches, output_dim=projection_dim
    )(positions)
    
    # Add position embeddings to patch embeddings
    # encoded_patches = patches + position_embedding
    encoded_patches = patches + tf.expand_dims(position_embedding, axis=0)
    
    # Create Transformer blocks
    for _ in range(transformer_layers):
        # Layer normalization 1
        x1 = LayerNormalization(epsilon=1e-6)(encoded_patches)
        
        # Multi-head attention
        attention_output = MultiHeadAttention(
            num_heads=num_heads, key_dim=projection_dim // num_heads, dropout=dropout_rate
        )(x1, x1)
        
        # Skip connection 1
        x2 = Add()([attention_output, encoded_patches])
        
        # Layer normalization 2
        x3 = LayerNormalization(epsilon=1e-6)(x2)
        
        # MLP
        x3 = mlp(x3, hidden_units=[projection_dim * 2, projection_dim], dropout_rate=dropout_rate)
        
        # Skip connection 2
        encoded_patches = Add()([x3, x2])
    
    # Layer normalization
    representation = LayerNormalization(epsilon=1e-6)(encoded_patches)
    
    # Global average pooling
    representation = layers.GlobalAveragePooling1D()(representation)
    
    # MLP head
    features = mlp(representation, hidden_units=mlp_head_units, dropout_rate=dropout_rate)
    
    # Output layer for regression
    outputs = Dense(1, activation="linear")(features)
    
    # Create the model
    model = Model(inputs=inputs, outputs=outputs)
    return model

In [ ]:
# model = create_vit_model(*default_params)

In [ ]:
path_best_model = f'/home/ht21074/Auto_box_apples/Auto_box_apples/Source_folder/Bayesian_optimisation_files/Bayes_opt_2026/{Code_run_ID}_ViT_best_model_params{type}.keras'


In [11]:

def evaluate_vit_model(
    patch_size, projection_dim, transformer_layers, num_heads, mlp_head_units, dropout_rate
):
    global best_loss
    global path_best_model

    clear_session()
    
    try:
        if testing == True:
            type = 'brix'
            X_train, y_train, X_val, y_val, train_cultivars, validate_encoder = load_data(type)
            train_generator = data_generator_w_cultivar(X_train[:3], y_train[:3], train_cultivars[:3], batch_size, img_size=img_size)
            val_generator = data_generator_w_cultivar(X_val[:3], y_val[:3], validate_encoder[:3], batch_size, img_size=img_size)
            n_epochs = 2
        else:
            train_generator = data_generator_w_cultivar(X_train, y_train, train_cultivars, batch_size, img_size=img_size)
            val_generator = data_generator_w_cultivar(X_val, y_val, validate_encoder, batch_size, img_size=img_size)
            n_epochs = 50

        model = create_vit_model(
            patch_size=patch_size,
            projection_dim=projection_dim,
            transformer_layers=transformer_layers,
            num_heads=num_heads,
            mlp_head_units=list(mlp_head_units),
            dropout_rate=dropout_rate
        )

        # weight_decay_callback = tf.keras.callbacks.LambdaCallback(
        #     on_epoch_end=lambda epoch, logs: [
        #         w.assign(w * (1 - 1e-5)) 
        #         for w in model.trainable_weights 
        #         if 'kernel' in w.name
        #     ]
        # )
        
        optimizer = Adam(
            learning_rate=1e-3,
            weight_decay=1e-5
            )

        model.compile(optimizer=optimizer, loss="mean_squared_error", metrics=["mae"])

        early_stopping = tf.keras.callbacks.EarlyStopping(
                    monitor='val_mae',
                    min_delta=0.001,  # 最小变化阈值
                    patience=15,      # 如果验证MAE在15轮内没有改善，则停止训练
                    verbose=0,
                    mode='min',
                    restore_best_weights=True)  # 恢复最佳权重

       
        history = model.fit(
            train_generator,
            epochs=n_epochs,  # 增加训练轮次
            steps_per_epoch=len(X_train) // batch_size,
            validation_data=val_generator,
            validation_steps=len(X_val) // batch_size,
            callbacks=[
            early_stopping,
            reduce_lr,
            # weight_decay_callback
            
    ],
)
        val_loss = np.mean(history.history["val_loss"])

        results_list.append([
            patch_size, projection_dim, transformer_layers, num_heads, mlp_head_units, dropout_rate, val_loss
            ])

        if val_loss < best_loss:
            best_loss = val_loss
            model.save(path_best_model)

        del model
        gc.collect()
        clear_session()

        return float(val_loss)

    except Exception as e:
        print(f"❌ Error: {e}")
        return float("inf")

In [12]:
@use_named_args(search_space)
def fitness_with_progress(patch_size,
    projection_dim,
    transformer_layers,
    num_heads,
    mlp_head_units,
    dropout_rate
):
                # Map parameters to name
                # param_names = ['patch_size', 'projection_dim', 'transformer_layers', 'num_heads', 'mlp_head_units', 'dropout_rate']
                # param_dict = dict(zip(param_names, params))
                # param_dict["mlp_head_units"] = parse_mlp_units(param_dict["mlp_head_units"])

                params = [patch_size, projection_dim, transformer_layers, num_heads, mlp_head_units, dropout_rate]
                result = evaluate_vit_model(
                    *params
                )
                tried_params.append(params)
                tried_scores.append(result)
                # str_units = "-".join(map(str, param_dict["mlp_head_units"]))
                

                
                # Save checkpoint
                try:
                   with open(checkpoint_path, 'wb') as f:
                       pickle.dump({
                           'x': tried_params,
                           'func_vals': tried_scores,
                           'results_list': results_list
                       }, f)
                   print(f"💾 Checkpoint saved to {checkpoint_path}")
                except Exception as e:
                  print(f"❌ Failed to save checkpoint: {e}")


                pbar.update(1)  # Update progress bar
                return float(result)


In [13]:
best_loss = np.inf


# Bayesian optimisation loop
for type in data_type:
    
    clear_session()
    print(f'Running optimisation for {type}')
    X_train, y_train, X_val, y_val, train_cultivars, validate_encoder = load_data(type)

    model_checkpoint_cb = tf.keras.callbacks.ModelCheckpoint(
                    filepath=f"{save_file_path}_{Code_run_ID}_model_file_{type}_vit.keras", 
                    monitor="val_mae", mode="min", 
                    save_best_only=True,
                    save_weights_only=False,
                    verbose=1)

    
    if testing == True:
        n_calls = 12
    else:
        n_calls = 100

    # Load checkpoint if exists
    checkpoint_path = f"{training_data_path}Bayes_opt_2026/{Code_run_ID}ViT_bayesian_optimization_checkpoint_{type}.pkl"
    
    os.makedirs(training_data_path, exist_ok=True)

    if os.path.exists(checkpoint_path):
        with open(checkpoint_path, 'rb') as f:
            checkpoint = pickle.load(f)
            tried_params = checkpoint['x']
            tried_scores = checkpoint['func_vals']
            results_list = checkpoint['results_list']
        print(f"🔁 Loaded checkpoint with {len(tried_params)} previous evaluations.")
    else:
        tried_params = []
        tried_scores = []
        results_list = []

    with tqdm(total=n_calls, desc="Optimisation Progress") as pbar:
        try: 
            
            # Perform Bayesian optimization       
            search_result = gp_minimize(
                func=fitness_with_progress,   
                dimensions=search_space,
                acq_func='EI',    #  'gp_hedge'       
                n_calls=n_calls,
                random_state=779,
                x0=tried_params if tried_params else [default_params],
                y0=tried_scores if tried_scores else None)

            # Save results to Excel
            df_results = pd.DataFrame(results_list, columns=[
                'patch_size',
                'projection_dim',
                'transformer_layers',
                'num_heads',
                'mlp_head_units',
                'dropout_rate',
                'val_loss'
            ])

            df_results.to_excel(f"{training_data_path}{Code_run_ID}_bayesian_optimization_results_{type}_ViT.xlsx", index=False)
            df_results.to_pickle(f"{training_data_path}{Code_run_ID}_bayesian_optimization_results_{type}_ViT.pkl")

            # Print best result summary
            print(f'✅ Best Validation Loss for {type}: {search_result.fun}')
            print(f'🏆 Best Parameters for {type}:')
            print(f'   patch_size = {search_result.x[0]}')
            print(f'   projection_dim = {search_result.x[1]}')
            print(f'   transformer_layers = {search_result.x[2]}')
            print(f'   num_heads = {search_result.x[3]}')
            print(f'   mlp_head_units = {search_result.x[4]}')
            print(f'   dropout_rate = {search_result.x[5]}')

        except Exception as e:
            print(f"❌ An error occurred during optimization for {type}: {e}")
            traceback.print_exc()
            continue


Running optimisation for starch
🔁 Loaded checkpoint with 11 previous evaluations.


Optimisation Progress:   0%|          | 0/12 [00:00<?, ?it/s]

2026-02-14 14:26:13.665199: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1929] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 79086 MB memory:  -> device: 0, name: NVIDIA A100 80GB PCIe, pci bus id: 0000:65:00.0, compute capability: 8.0


Epoch 1/2


2026-02-14 14:26:24.412006: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:454] Loaded cuDNN version 8907
2026-02-14 14:26:26.442854: I external/local_xla/xla/service/service.cc:168] XLA service 0x7f3199288990 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
2026-02-14 14:26:26.442898: I external/local_xla/xla/service/service.cc:176]   StreamExecutor device (0): NVIDIA A100 80GB PCIe, Compute Capability 8.0
2026-02-14 14:26:26.451346: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1771079186.582651 3188235 device_compiler.h:186] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


...

💾 Checkpoint saved to /home/ht21074/Auto_box_apples/Auto_box_apples/Source_folder/Bayesian_optimisation_files/Bayes_opt_2026/2026-02-14run_29_ViT_bays_opt_40pxViT_bayesian_optimization_checkpoint_starch.pkl
Epoch 1/2
391/391 [==============================] - 26s 34ms/step - loss: 3.2446 - mae: 1.3308 - val_loss: 2.0425 - val_mae: 1.3089 - lr: 0.0010
Epoch 2/2
391/391 [==============================] - 12s 30ms/step - loss: 1.4274 - mae: 0.9424 - val_loss: 2.3727 - val_mae: 1.4097 - lr: 0.0010


Optimisation Progress:  33%|███▎      | 4/12 [03:11<06:08, 46.12s/it]

💾 Checkpoint saved to /home/ht21074/Auto_box_apples/Auto_box_apples/Source_folder/Bayesian_optimisation_files/Bayes_opt_2026/2026-02-14run_29_ViT_bays_opt_40pxViT_bayesian_optimization_checkpoint_starch.pkl
Epoch 1/2
391/391 [==============================] - 31s 37ms/step - loss: 4.6700 - mae: 1.4956 - val_loss: 2.1298 - val_mae: 1.1836 - lr: 0.0010
Epoch 2/2
133/391 [=========>....................] - ETA: 8s - loss: 1.6299 - mae: 1.0037

Optimisation Progress:  33%|███▎      | 4/12 [03:47<07:35, 56.95s/it]


KeyboardInterrupt: 

In [1]:
import pickle

with open("/home/ht21074/Auto_box_apples/Auto_box_apples/Source_folder/Bayesian_optimisation_files/Bayes_opt_2026/bayesian_optimization_checkpoint_starch.pkl", "rb") as f:
    obj = pickle.load(f)
print(type(obj))

<class 'dict'>


In [3]:
print(obj.keys())

dict_keys(['x', 'func_vals', 'results_list'])


In [5]:
import pprint
pprint.pprint(obj)

{'func_vals': [inf,
               inf,
               inf,
               inf,
               inf,
               inf,
               inf,
               inf,
               inf,
               inf,
               inf,
               inf,
               inf,
               inf,
               inf,
               inf,
               inf,
               inf,
               inf,
               inf,
               inf],
 'results_list': [[9, 199, 4, 3, '256-128', 0.411049943926979, inf],
                  [9, 199, 4, 3, '256-128', 0.411049943926979, inf],
                  [7, 109, 5, 3, '128-64', 0.1678213614105492, inf],
                  [6, 154, 4, 2, '256-128-64', 0.3488038045944766, inf],
                  [9, 199, 4, 3, '256-128', 0.411049943926979, inf],
                  [9, 199, 4, 3, '256-128', 0.411049943926979, inf],
                  [7, 109, 5, 3, '128-64', 0.1678213614105492, inf],
                  [6, 154, 4, 2, '256-128-64', 0.3488038045944766, inf],
                  [

In [ ]:
import pandas as pd
import pickle
import numpy as np


with open('/home/ht21074/Auto_box_apples/Auto_box_apples/Source_folder/Bayesian_optimisation_files/2026-02-11run_29_2D_CNN_bays_opt_40px_2D_CNN_bayesian_optimization_checkpoint_brix.pkl', 'rb') as f:
    file = pickle.load(f)
print(file)

In [ ]:
# Recover 2D CNN parameters via checkpoints

import pickle
import numpy as np

checkpoint_path = "/home/ht21074/Auto_box_apples/Auto_box_apples/Source_folder/Bayesian_optimisation_files/2026-02-11run_29_2D_CNN_bays_opt_40px_2D_CNN_bayesian_optimization_checkpoint_brix.pkl"

with open(checkpoint_path, "rb") as f:
    checkpoint = pickle.load(f)

params = checkpoint["x"]
scores = checkpoint["func_vals"]

best_index = np.argmin(scores)
best_params = params[best_index]
best_score = scores[best_index]

print("Best params:", best_params)
print("Best score:", best_score)


Best params: [4, 97, 351, 496, 2025, 803, 2, 1, 0.451685394785047, 2]
Best score: 41.28987018878643


In [ ]:
# Recover 2D CNN parameters via checkpoints

import pickle
import numpy as np

checkpoint_path = "Source_folder/Bayesian_optimisation_files/2026-02-15run_29_2D_CNN_bays_opt_40px_2D_CNN_bayesian_optimization_checkpoint_firmness_converted.pkl.pkl"

with open(checkpoint_path, "rb") as f:
    checkpoint = pickle.load(f)

params = checkpoint["x"]
scores = checkpoint["func_vals"]

best_index = np.argmin(scores)
best_params = params[best_index]
best_score = scores[best_index]

print("Best params:", best_params)
print("Best score:", best_score)


FileNotFoundError: [Errno 2] No such file or directory: 'Source_folder/Bayesian_optimisation_files/2026-02-15run_29_2D_CNN_bays_opt_40px_2D_CNN_bayesian_optimization_checkpoint_firmness_converted.pkl'

In [42]:
print(checkpoint["x"])
print(checkpoint["func_vals"])
print(len(scores))
print(scores[-1])

[[3, 64, 128, 256, 512, 512, 1, 0.15, 1], [4, 97, 351, 496, 2025, 803, 2, 0.1928048321256467, 2], [3, 33, 113, 499, 981, 772, 1, 0.3234957214183365, 2], [1, 91, 272, 66, 2044, 249, 2, 0.10477069580013461, 2], [4, 86, 77, 804, 580, 133, 1, 0.10057593529712348, 1], [4, 84, 332, 957, 1950, 928, 1, 0.22218527046722947, 2], [3, 85, 444, 528, 374, 694, 2, 0.4235763544598753, 1], [5, 99, 270, 829, 553, 844, 2, 0.4704138087919506, 2], [4, 94, 411, 1001, 1373, 480, 1, 0.32137823990168585, 2], [4, 84, 185, 479, 1682, 624, 1, 0.24012328063417304, 2], [4, 77, 88, 728, 814, 993, 1, 0.33146389255871594, 2], [5, 110, 492, 766, 237, 945, 2, 0.5, 2], [1, 62, 382, 161, 367, 1024, 1, 0.222440001050492, 1], [1, 95, 303, 32, 564, 1024, 1, 0.40023718495157923, 1], [3, 39, 406, 81, 626, 965, 1, 0.21212553254742425, 1], [1, 108, 32, 32, 390, 1024, 1, 0.5, 1], [2, 90, 102, 134, 333, 861, 2, 0.18415543917531088, 1], [5, 52, 186, 750, 505, 126, 2, 0.1, 1], [2, 95, 512, 591, 170, 138, 1, 0.437410719817128, 1], [5

In [45]:
print(checkpoint)


{'x': [[3, 64, 128, 256, 512, 512, 1, 0.15, 1], [4, 97, 351, 496, 2025, 803, 2, 0.1928048321256467, 2], [3, 33, 113, 499, 981, 772, 1, 0.3234957214183365, 2], [1, 91, 272, 66, 2044, 249, 2, 0.10477069580013461, 2], [4, 86, 77, 804, 580, 133, 1, 0.10057593529712348, 1], [4, 84, 332, 957, 1950, 928, 1, 0.22218527046722947, 2], [3, 85, 444, 528, 374, 694, 2, 0.4235763544598753, 1], [5, 99, 270, 829, 553, 844, 2, 0.4704138087919506, 2], [4, 94, 411, 1001, 1373, 480, 1, 0.32137823990168585, 2], [4, 84, 185, 479, 1682, 624, 1, 0.24012328063417304, 2], [4, 77, 88, 728, 814, 993, 1, 0.33146389255871594, 2], [5, 110, 492, 766, 237, 945, 2, 0.5, 2], [1, 62, 382, 161, 367, 1024, 1, 0.222440001050492, 1], [1, 95, 303, 32, 564, 1024, 1, 0.40023718495157923, 1], [3, 39, 406, 81, 626, 965, 1, 0.21212553254742425, 1], [1, 108, 32, 32, 390, 1024, 1, 0.5, 1], [2, 90, 102, 134, 333, 861, 2, 0.18415543917531088, 1], [5, 52, 186, 750, 505, 126, 2, 0.1, 1], [2, 95, 512, 591, 170, 138, 1, 0.437410719817128, 

In [ ]:
import pickle
from skopt import Optimizer
from skopt.space import Integer, Real

# -----------------------------
# Load old checkpoint
# -----------------------------
old_path = "/home/ht21074/Auto_box_apples/Auto_box_apples/Source_folder/Bayesian_optimisation_files/2026-02-15run_29_2D_CNN_bays_opt_40px_2D_CNN_bayesian_optimization_checkpoint_starch.pkl"

with open(old_path, "rb") as f:
    old = pickle.load(f)

x_iters = old["x"]          # 100 parameter vectors
func_vals = old["func_vals"]  # 100 scores

print("Loaded:", len(x_iters), "params,", len(func_vals), "scores")


# -----------------------------
# Rebuild optimizer with your NEW search space
# -----------------------------
search_space = [
    Integer(1, 5, name="num_layers"),
    Integer(32, 124, name="filters1"),
    Integer(32, 512, name="filters2"),
    Integer(32, 1024, name="filters3"),
    Integer(32, 2048, name="filters4"),
    Integer(32, 1024, name="filters5"),
    Integer(1, 2, name="kernel_size"),
    Real(0.1, 0.5, name="dropout"),
    Integer(1, 2, name="pool_size"),
]

opt = Optimizer(search_space, random_state=779)

# -----------------------------
# Replay all 100 trials
# -----------------------------
for x, y in zip(x_iters, func_vals):
    opt.tell(x, y)

print("Optimizer rebuilt with", len(opt.Xi), "trials")


# -----------------------------
# Save new checkpoint in NEW format
# -----------------------------
new_checkpoint = {
    "optimizer": opt,
    "params": x_iters,
    "scores": func_vals
}

new_path = old_path.replace(".pkl", "_converted.pkl")

with open(new_path, "wb") as f:
    pickle.dump(new_checkpoint, f)

print("Saved converted checkpoint to:", new_path)

KeyError: 'x'

In [15]:
# Recover 2D CNN parameters via checkpoints

import pickle
import numpy as np

checkpoint_path = "/home/ht21074/Auto_box_apples/Auto_box_apples/Source_folder/Bayesian_optimisation_files/Bayes_opt_2026/2026-02-18_hybrid_BO_checkpoint_brix.pkl"
with open(checkpoint_path, "rb") as f:
    checkpoint1 = pickle.load(f)

checkpoint_path = "/home/ht21074/Auto_box_apples/Auto_box_apples/Source_folder/Bayesian_optimisation_files/Bayes_opt_2026/2026-02-19_hybrid_BO_checkpoint_brix.pkl"
with open(checkpoint_path, "rb") as g:
    checkpoint2 = pickle.load(g)

checkpoint = checkpoint1.copy()
checkpoint.update(checkpoint2)

print(checkpoint)



params = checkpoint["params"]
scores = checkpoint["scores"]

best_index = np.argmin(scores)
best_params = params[best_index]
best_score = scores[best_index]

print("Best params:", best_params)
print("Best score:", best_score)
print(len(scores))


{'optimizer': <skopt.optimizer.optimizer.Optimizer object at 0x7f409cd48610>, 'params': [[3, 118, 88, 137, 5, 0.3332874579452343, 2, 2, 9, 160, 1, (128, 64)], [2, 81, 98, 109, 4, 0.28660285344585745, 1, 4, 6, 32, 8, (128, 64)], [3, 10, 92, 199, 4, 0.12796381828227676, 3, 2, 3, 128, 1, (128, 64)], [3, 96, 83, 241, 5, 0.3708969328787224, 1, 2, 8, 128, 4, (256, 128)], [2, 152, 102, 176, 4, 0.33660191317824384, 3, 4, 6, 128, 4, (256, 128)], [2, 34, 88, 164, 4, 0.18379572855920362, 3, 4, 6, 224, 2, (256, 128)], [2, 96, 46, 133, 4, 0.27902302128432405, 1, 2, 9, 192, 2, (128, 64)], [2, 68, 124, 75, 4, 0.33804943075177274, 3, 1, 4, 128, 2, (256, 128)], [2, 64, 34, 154, 3, 0.19336381826491433, 1, 1, 10, 192, 2, (256, 128)], [2, 50, 127, 248, 4, 0.33417591429958193, 2, 4, 10, 224, 8, (256, 128, 64)]], 'scores': [1.0289474725723267, 1.716632604598999, 1.1582067012786865, 1.1091818809509277, 1.1268881559371948, 1.0206897258758545, 0.9494684338569641, 1.098091959953308, 1.1994982957839966, 1.849835

In [14]:
from pprint import pprint

pprint(checkpoint["params"])

[[3, 118, 88, 137, 5, 0.3332874579452343, 2, 2, 9, 160, 1, (128, 64)],
 [2, 81, 98, 109, 4, 0.28660285344585745, 1, 4, 6, 32, 8, (128, 64)],
 [3, 10, 92, 199, 4, 0.12796381828227676, 3, 2, 3, 128, 1, (128, 64)],
 [3, 96, 83, 241, 5, 0.3708969328787224, 1, 2, 8, 128, 4, (256, 128)],
 [2, 152, 102, 176, 4, 0.33660191317824384, 3, 4, 6, 128, 4, (256, 128)],
 [2, 34, 88, 164, 4, 0.18379572855920362, 3, 4, 6, 224, 2, (256, 128)],
 [2, 96, 46, 133, 4, 0.27902302128432405, 1, 2, 9, 192, 2, (128, 64)],
 [2, 68, 124, 75, 4, 0.33804943075177274, 3, 1, 4, 128, 2, (256, 128)],
 [2, 64, 34, 154, 3, 0.19336381826491433, 1, 1, 10, 192, 2, (256, 128)],
 [2, 50, 127, 248, 4, 0.33417591429958193, 2, 4, 10, 224, 8, (256, 128, 64)]]
